# L4 — Independent sets: the first penalty

**2026 Quantum Accelerator · Day 1 · QUBO ladder · step 4 of 5 · ~20 minutes**

**New on this step:** objective and feasibility are *separated* inside one matrix. The diagonal
is the real objective (`−w_i`); the off-diagonal is a penalty `P` on every edge that enforces
"never two neighbours". This is the first **tunable parameter**, with a provable lower bound
(`P > max w`) and a visible failure when it is too small.

**The problem.** Pick nodes of a graph with the most total weight such that no two picked nodes
are neighbours. **Why it matters.** Conflict graphs — trains, frequencies, meeting rooms — and
the native problem of neutral-atom quantum hardware (Day 2). **Plan.** The slide instance, a
150-node unit-disk graph, then turn the penalty dial and watch the answer break.

---
## 1. The slide instance

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import helpers_ladder as h

labels = ["A", "B", "C", "D"]
points, w, edges, P = h.mwis_slide()          # 4-cycle A-B-C-D, weights 4, 6, 5, 2
P = 8                                         # <-- CHANGE ME: try 2, then 3, then re-run sections 2-4

fig, ax = plt.subplots(figsize=(3.6, 3.6))
for i, j in edges:
    ax.plot(*zip(points[i], points[j]), color="#888888", zorder=1)
ax.scatter(*points.T, s=900, color="#dbe9f6", edgecolors="#555555", zorder=2)
for k, (x_, y_) in enumerate(points):
    ax.text(x_, y_, f"{labels[k]}\nw={w[k]}", ha="center", va="center", fontsize=10)
ax.set_xlim(-0.4, 1.4); ax.set_ylim(-0.4, 1.4); ax.set_aspect("equal"); ax.axis("off")
plt.show()

Greedy takes B (6) and then D (2): total 8. The optimum is A and C: total 9. The penalty must
make the annealer prefer the second.

---
## 2. Q by hand

In [2]:
Q = -np.diag(w).astype(float)
for i, j in edges:
    Q[i, j] += P / 2; Q[j, i] += P / 2        # cross term = P on every edge
h.print_matrix(Q, labels, f"Q (P = {P})")

Q (P = 8)
       A   B   C   D
   A  -4   4   0   4
   B   4  -6   4   0
   C   0   4  -5   4
   D   4   0   4  -2


**What you see.** The diagonal is `−w`. Every edge adds `P/2` on both sides of the diagonal,
i.e. a cross term of `P`. The zeros at (A, C) and (B, D) are the **non-edges**: Q is as sparse as
the graph, unlike the dense Q of L2.

---
## 3. Every answer

In [3]:
table = h.exact_table(Q, labels)
table["weight"] = [int(w @ row.values[:4]) for _, row in table.iterrows()]
table["independent"] = [h.is_independent(edges, row.values[:4]) for _, row in table.iterrows()]
table

,A,B,C,D,energy,objective,weight,independent
0,1,0,1,0,-9,-9,9,True
1,0,1,0,1,-8,-8,8,True
2,0,1,0,0,-6,-6,6,True
3,0,0,1,0,-5,-5,5,True
4,1,0,0,0,-4,-4,4,True
5,0,1,1,0,-3,-3,11,False
6,0,0,0,1,-2,-2,2,True
7,1,1,0,0,-2,-2,10,False
8,0,0,0,0,0,0,0,True
9,0,0,1,1,1,1,7,False


**What you see.** The best row is A and C at −9. At a feasible point the penalty contributes
exactly 0, so the energy *is* minus the weight. Every infeasible row sits above −4. The rule in
one line: `P` must exceed the largest weight (here 6), so that dropping a node always beats
keeping a conflict.

---
## 4. The same Q from PyQUBO

In [4]:
from pyqubo import Binary, Constraint, Placeholder

x = [Binary(l) for l in labels]
reward = -sum(int(wi) * xi for wi, xi in zip(w, x))
conflicts = sum(Constraint(x[i] * x[j], label=f"edge_{labels[i]}{labels[j]}") for i, j in edges)
model = (reward + Placeholder("P") * conflicts).compile()
qubo, offset = model.to_qubo(feed_dict={"P": P})
print(np.allclose(Q, h.symmetric_from_qubo_dict(qubo, labels)), offset)   # True 0.0

True 0.0


**What you see.** `Placeholder("P")` makes the penalty a *dial*: compile once, sweep later. This
is the pattern notebooks 02 and 04 use. `Constraint` labels each edge so a decoder can report which
ones were violated.

**The penalty dial on the slide instance.**

In [5]:
for P_try in (1, 2, 3, 4, 8, 20):
    Qp = h.mwis_Q(w, edges, P_try)
    t = h.exact_table(Qp, labels)
    best = t[t.energy == t.energy.iloc[0]]
    arg = [tuple(int(v) for v in r[:4]) for r in best.values]
    print(f"P = {P_try:>2}: argmin {arg}  energy {t.energy.iloc[0]}  "
          f"independent: {[h.is_independent(edges, a) for a in arg]}")

P =  1: argmin [(1, 1, 1, 0), (1, 1, 1, 1)]  energy -13  independent: [False, False]
P =  2: argmin [(1, 1, 1, 0)]  energy -11  independent: [False]
P =  3: argmin [(1, 0, 1, 0), (1, 1, 1, 0)]  energy -9  independent: [True, False]
P =  4: argmin [(1, 0, 1, 0)]  energy -9  independent: [True]
P =  8: argmin [(1, 0, 1, 0)]  energy -9  independent: [True]
P = 20: argmin [(1, 0, 1, 0)]  energy -9  independent: [True]


**What you see.** P = 1 and 2 pick an infeasible set (the penalty is cheaper than dropping a
node); P = 3 is a tie between a feasible and an infeasible set; from P = 4 the answer A + C is
unique. The per-edge bound `P > min(w_i, w_j)` on the worst edge (B–C: 5) explains why 4 is already
enough here; the global rule `P > 6` is the safe one.

---
## 5. Now too big for a slide: a unit-disk graph with 150 nodes

In [6]:
points_b, w_big, edges_big = h.mwis_instance(n=150, radius=0.12, seed=1)   # <-- CHANGE ME: radius 0.10 / 0.15
P_big = int(w_big.max()) + 1                                               # 11
Q_big = h.mwis_Q(w_big, edges_big, P_big)
ss, dt = h.anneal(Q_big)
stats = h.mwis_stats(ss, w_big, edges_big, 150)
ref = h.cpsat_mwis(w_big, edges_big)

print(f"{len(w_big)} nodes, {len(edges_big)} edges, P = {P_big}")
print(f"SA: {stats['feasible_pct']:.0f} % of reads feasible, best feasible weight {stats['best_feasible_value']}   ({dt:.2f} s)")
print(f"CP-SAT: optimum {ref['value']:.0f}, status {ref['status']}, {ref['seconds']:.2f} s")
h.draw_disk_graph(points_b, edges_big, stats["best_x"], radius=0.12, title="Best feasible independent set (SA)")

150 nodes, 432 edges, P = 11
SA: 99 % of reads feasible, best feasible weight 284   (0.53 s)
CP-SAT: optimum 284, status OPTIMAL, 0.01 s


**What you see.** The annealer finds the proven optimum — and the exact solver is faster *and*
proves it. Say so plainly. The unit-disk picture is the Day-2 bridge: in a neutral-atom machine
two atoms closer than the blockade radius cannot both be excited, so the penalty is physics, not a number.

---
## 5b. The penalty dial

In [7]:
rows = h.penalty_sweep_mwis(w_big, edges_big, Ps=(1, 2, 3, 5, 8, 11, 15, 20, 40, 80))   # <-- CHANGE ME
print(f"{'P':>4}{'feasible':>10}{'best':>7}")
for r in rows:
    print(f"{r['P']:>4}{r['feasible_pct']:>9.0f}%{str(r['best_value']) if r['best_value'] else '-':>7}")
h.plot_penalty_sweep(rows, reference=ref["value"])

   P  feasible   best
   1        0%      -
   2        0%      -
   3        0%      -
   5        0%      -
   8        0%      -
  11       99%    284
  15      100%    284
  20      100%    281
  40      100%    262
  80      100%    223


**What you see.** Nothing is feasible until `P` exceeds the largest weight (10); then quality
*decays* as `P` grows, because the landscape is dominated by the penalty and the annealer settles
for small, safe sets. The smallest provably sufficient `P` is the right choice. This is the general
shape of every λ sweep today (notebook 02 shows the same curve for knapsack).

---
## 6. What changed on this step

- Objective (diagonal) and rule (off-diagonal) live in one matrix, one penalty per edge.
- `P > max w` is a provable bound; below it, the annealer's best answer is illegal.
- Above it, a larger `P` only makes the answer worse: the first legality-versus-quality trade-off.
- Hand-built Q and PyQUBO (with `Placeholder`) agree exactly.
- Next, **L5**: a *global* rule ("at most 7 kg") needs slack bits, and the encoding starts to cost.

---
## Stretch

**A.** Replace the unit-disk graph by `nx.gnp_random_graph(150, 0.05)` with the same weights. Is the
`P` threshold still at max w? (It is — the bound does not depend on the graph.)

**B.** Cardinality: add "pick exactly 20 nodes" as a second squared penalty and watch feasibility
collapse — a preview of notebook 05's right-hand panel.

**C.** The Rydberg question: which of the two graphs (unit-disk or G(n, p)) could a neutral-atom
machine take natively, and why?